In [1]:
!pip install -q groq requests

In [2]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
FDC_API_KEY = userdata.get("FDC_API_KEY")

print("API keys loaded successfully!")

API keys loaded successfully!


In [5]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-120b"

print("NutriAI model initialized successfully!")

NutriAI model initialized successfully!


In [6]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Introduce yourself as a Nutrient AI Agent."
        }
    ]
)

print(response.choices[0].message.content)

Hello! I’m **Nutrient AI**, your dedicated Nutrient AI Agent. I specialize in translating scientific research and data about nutrition, health, and wellness into clear, practical insights you can use daily. Whether you’re curious about the latest findings on micronutrients, need help interpreting dietary guidelines, or want evidence‑based tips for optimizing your meals, I’m here to help you make informed, health‑forward choices. Feel free to ask me anything related to nutrition, food science, or healthy eating!


In [7]:
import requests

FDC_BASE_URL = "https://api.nal.usda.gov/fdc/v1"

print("USDA API configuration ready!")

USDA API configuration ready!


In [9]:
def search_food(food_name):

    url = f"{FDC_BASE_URL}/foods/search"

    params = {
        "query": food_name,
        "api_key": FDC_API_KEY,
        "pageSize": 5
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        return data.get("foods", [])

    else:
        print("API Error:", response.status_code)
        return []

In [10]:
foods = search_food("Apple")

for food in foods:
    print("Food:", food["description"])
    print("FDC ID:", food["fdcId"])
    print("--------------------")

Food: APPLE
FDC ID: 454004
--------------------
Food: APPLE
FDC ID: 2117388
--------------------
Food: APPLE
FDC ID: 2124902
--------------------
Food: Apple, candied
FDC ID: 2709294
--------------------
Food: Apple, raw
FDC ID: 2709215
--------------------


In [11]:
def get_food_details(fdc_id):

    url = f"{FDC_BASE_URL}/food/{fdc_id}"

    params = {
        "api_key": FDC_API_KEY
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:
        return response.json()

    else:
        print("Error:", response.status_code)
        return None

In [12]:
food_data = get_food_details(foods[0]["fdcId"])

print("Food:", food_data["description"])

for nutrient in food_data.get("foodNutrients", []):

    name = nutrient.get("nutrient", {}).get("name")
    amount = nutrient.get("amount")
    unit = nutrient.get("nutrient", {}).get("unitName")

    if amount is not None:
        print(name, ":", amount, unit)

Food: APPLE
Fatty acids, total saturated : 0.0 g
Calcium, Ca : 0.0 mg
Vitamin C, total ascorbic acid : 3.1 mg
Cholesterol : 0.0 mg
Iron, Fe : 0.23 mg
Sodium, Na : 0.0 mg
Vitamin A, IU : 65.0 IU
Total Sugars : 10.39 g
Energy : 52.0 kcal
Potassium, K : 110.0 mg
Protein : 0.0 g
Fatty acids, total trans : 0.0 g
Fiber, total dietary : 3.2 g
Carbohydrate, by difference : 14.29 g
Total lipid (fat) : 0.65 g


In [13]:
import json

def understand_meal(meal):

    prompt = f"""
    You are a food quantity estimation assistant.

    Analyze this meal: {meal}

    Identify each food item separately.

    Rules:
    1. Identify food names and quantities.
    2. Convert serving quantities into estimated grams.
    3. For one medium bowl of cooked rice, assume 150 grams.
    4. For one omelette without egg count, assume 2 eggs.
    5. For one omelette, estimate 5 grams cooking oil.
    6. Break mixed dishes into ingredients where appropriate.
    7. Never invent a missing main food item.
    8. If the user says only "1 bowl" without identifying
       what is inside it, ask for clarification.

    Return JSON only in this format:

    {{
      "clarification_needed": false,
      "question": "",
      "items": [
        {{
          "food": "cooked white rice",
          "search_term": "cooked white rice",
          "quantity": "1 medium bowl",
          "grams": 150,
          "assumption": "Estimated serving size"
        }}
      ]
    }}
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    return json.loads(response.choices[0].message.content)

In [14]:
meal = "1 bowl rice + 1 omelette"

meal_data = understand_meal(meal)

print(json.dumps(meal_data, indent=4))

{
    "clarification_needed": false,
    "question": "",
    "items": [
        {
            "food": "cooked white rice",
            "search_term": "cooked white rice",
            "quantity": "1 medium bowl",
            "grams": 150,
            "assumption": "Estimated serving size"
        },
        {
            "food": "egg",
            "search_term": "egg, whole, raw",
            "quantity": "2 eggs",
            "grams": 100,
            "assumption": "2 eggs assumed for omelette"
        },
        {
            "food": "cooking oil",
            "search_term": "cooking oil",
            "quantity": "5 grams",
            "grams": 5,
            "assumption": "5g oil for cooking omelette"
        }
    ]
}


In [15]:
def extract_nutrients(food_data):

    nutrients = {}

    for item in food_data.get("foodNutrients", []):

        if "nutrient" in item:
            name = item["nutrient"].get("name", "")
            amount = item.get("amount")
            unit = item["nutrient"].get("unitName", "")

        else:
            name = item.get("nutrientName", "")
            amount = item.get("value")
            unit = item.get("unitName", "")

        if amount is None:
            continue

        name = name.lower()

        if "energy" in name and unit.lower() in ["kcal", "calorie"]:
            nutrients["Calories"] = amount

        elif name == "protein":
            nutrients["Protein"] = amount

        elif "carbohydrate, by difference" in name:
            nutrients["Carbs"] = amount

        elif name == "total lipid (fat)":
            nutrients["Fat"] = amount

        elif name == "fiber, total dietary":
            nutrients["Fiber"] = amount

    return nutrients

In [16]:
def calculate_meal_nutrition(meal_data):

    totals = {
        "Calories": 0,
        "Protein": 0,
        "Carbs": 0,
        "Fat": 0,
        "Fiber": 0
    }

    breakdown = []

    for item in meal_data["items"]:

        foods = search_food(item["search_term"])

        if not foods:
            print("Food not found:", item["food"])
            continue

        selected = foods[0]

        details = get_food_details(selected["fdcId"])

        if not details:
            continue

        nutrients = extract_nutrients(details)

        grams = item["grams"]

        scaled = {}

        for nutrient in totals:

            value = nutrients.get(nutrient, 0)

            scaled[nutrient] = value * grams / 100

            totals[nutrient] += scaled[nutrient]

        breakdown.append({
            "Food": item["food"],
            "Quantity": item["quantity"],
            "Grams": grams,
            "Nutrition": scaled,
            "FDC_ID": selected["fdcId"]
        })

    return totals, breakdown

In [17]:
def analyze_meal(meal):

    meal_data = understand_meal(meal)

    if meal_data["clarification_needed"]:
        print("Please clarify:", meal_data["question"])
        return

    totals, breakdown = calculate_meal_nutrition(meal_data)

    print("\nFOOD-WISE BREAKDOWN")

    for item in breakdown:
        print("\nFood:", item["Food"])
        print("Quantity:", item["Quantity"])
        print("Nutrition:", item["Nutrition"])

    print("\nTOTAL NUTRITION")

    for nutrient, value in totals.items():
        print(nutrient, ":", round(value, 2))

    return totals, breakdown

In [19]:
analyze_meal("1 bowl rice + 1 omelette")


FOOD-WISE BREAKDOWN

Food: cooked white rice
Quantity: 1 medium bowl
Nutrition: {'Calories': 214.5, 'Protein': 4.29, 'Carbs': 47.145, 'Fat': 0.0, 'Fiber': 0.0}

Food: egg
Quantity: 2 eggs
Nutrition: {'Calories': 513.0, 'Protein': 6.41, 'Carbs': 57.69, 'Fat': 28.85, 'Fiber': 3.2}

Food: cooking oil
Quantity: 5 grams
Nutrition: {'Calories': 7.2, 'Protein': 0.063, 'Carbs': 1.545, 'Fat': 0.16200000000000003, 'Fiber': 0.08}

TOTAL NUTRITION
Calories : 734.7
Protein : 10.76
Carbs : 106.38
Fat : 29.01
Fiber : 3.28


({'Calories': 734.7,
  'Protein': 10.763,
  'Carbs': 106.38000000000001,
  'Fat': 29.012,
  'Fiber': 3.2800000000000002},
 [{'Food': 'cooked white rice',
   'Quantity': '1 medium bowl',
   'Grams': 150,
   'Nutrition': {'Calories': 214.5,
    'Protein': 4.29,
    'Carbs': 47.145,
    'Fat': 0.0,
    'Fiber': 0.0},
   'FDC_ID': 356556},
  {'Food': 'egg',
   'Quantity': '2 eggs',
   'Grams': 100,
   'Nutrition': {'Calories': 513.0,
    'Protein': 6.41,
    'Carbs': 57.69,
    'Fat': 28.85,
    'Fiber': 3.2},
   'FDC_ID': 2575290},
  {'Food': 'cooking oil',
   'Quantity': '5 grams',
   'Grams': 5,
   'Nutrition': {'Calories': 7.2,
    'Protein': 0.063,
    'Carbs': 1.545,
    'Fat': 0.16200000000000003,
    'Fiber': 0.08},
   'FDC_ID': 2709559}])